# 04 - Prepare Histology Stack for EM-LDDMM

After tissue sections have been extracted and renamed (notebooks 01-02), this
notebook prepares the image stack for the EM-LDDMM registration pipeline:

1. Define physical voxel coordinates
2. Clean up previous JSON sidecars
3. Generate `samples.tsv` (with missing-slice detection) and JSON sidecar files

**Prerequisites:**
- Renamed tissue tiles from **notebook 01**
- Docker clones `https://github.com/twardlab/emlddmm.git`, installs modern runtime dependencies, and adds it to `PYTHONPATH`
- For a local editable install, use `pip install -e .`
- Local editable installs need the external `emlddmm` checkout for notebook 04 sidecars and notebook 05 registration

In [1]:
import importlib.util
import sys
from pathlib import Path

# Local non-Docker runs need the upstream checkout on sys.path/PYTHONPATH.
EMLDDMM_REPO = Path("../emlddmm").expanduser().resolve()
if importlib.util.find_spec("histsetup") is None and EMLDDMM_REPO.exists():
    sys.path.insert(0, str(EMLDDMM_REPO))

if importlib.util.find_spec("histsetup") is None:
    raise ModuleNotFoundError(
        "Could not import histsetup from the upstream emlddmm checkout. "
        "For local runs, use: git clone https://github.com/twardlab/emlddmm.git ../emlddmm; "
        "python -m pip install -c constraints.txt \"h5py>=3.10\" \"nibabel>=5.0\"; "
        "python -m pip install --index-url https://download.pytorch.org/whl/cpu "
        "\"torch==2.10.0+cpu\"; then restart or rerun this cell."
    )

from wsi_pipeline.emlddmm_prep import (
    make_samples_tsv,
    set_up_hist_for_emlddmm,
    nuke_json_sidecars,
)

%load_ext autoreload
%autoreload 2

ModuleNotFoundError: Could not import histsetup from the upstream emlddmm checkout. For local runs, use: git clone https://github.com/twardlab/emlddmm.git ../emlddmm; python -m pip install -c constraints.txt "h5py>=3.10" "nibabel>=5.0"; python -m pip install --index-url https://download.pytorch.org/whl/cpu "torch==2.10.0+cpu"; then restart or rerun this cell.

## Step 1: Define Physical Coordinates

The highest-resolution image has a known pixel size (from the microscope metadata).
At a given pyramid level, the pixel size doubles with each level:

```
pixel_size_at_level = base_pixel_size * 2^level
```

The `dv` vector is `[dx, dy, dz]` in **micrometers** (XYZ order), where `dz` is
the physical spacing between collected slices (e.g. 16 um for 10-um slices
collected every 10th section with some mounting gap).

In [ ]:
# Docker defaults line up with notebook 01's demo data and TIFF tile outputs.
# subject_wsi_dir = Path("/data/input").expanduser().resolve()
# output_dir = Path("/output/tissue_sections").expanduser().resolve()
# subject_wsi_dir = Path("/cis/home/dpadova/Documents/temporal_bone_project/notebook_flatfile_smoke_level7/tissue_sections").expanduser().resolve()
subject_wsi_dir = Path("/cis/home/dpadova/Documents/temporal_bone_project/data/FlatFiles").expanduser().resolve()
output_dir = Path("/cis/home/dpadova/Documents/temporal_bone_project/notebook_flatfile_smoke_level7/tissue_sections").expanduser().resolve()


# Physical pixel size of the highest-resolution image (ZYX, micrometers)
base_scale = [1.0, 0.27385655, 0.27385828]  # Update for your microscope

# Pyramid level used for the extracted tiles
pyramid_level = 7  # Update for your data

# Compute pixel size at this level (XYZ order for dv)
level_07_dv = [
    base_scale[1] * pow(2, pyramid_level),  # dx
    base_scale[2] * pow(2, pyramid_level),  # dy
    16.0,                                   # dz (inter-slice spacing in um)
]

print(f"Voxel size at level {pyramid_level}: {level_07_dv} um (XYZ)")

# Full configuration for set_up_hist_for_emlddmm
histsetup_config = {
    "subject_dir": subject_wsi_dir,
    "output_dir": output_dir,
    "species_name": "Macaque",  # Update for your species
    "ext": ".tif",
    "slice_down": 1,      # 1 = use all slices (no further downsampling)
    "res_down": 1,        # 1 = no spatial downsampling
    "max_slice": None,    # infer from data
    "dv": level_07_dv,
    "space": "right-inferior-posterior",
}

print(f"Input dir:  {subject_wsi_dir}")
print(f"Output dir: {output_dir}")
print("\nConfig:")
for k, v in histsetup_config.items():
    print(f"  {k}: {v}")

Voxel size at level 7: [35.0536384, 35.05385984, 16.0] um (XYZ)
Input dir:  /cis/home/dpadova/Documents/temporal_bone_project/data/FlatFiles
Output dir: /cis/home/dpadova/Documents/temporal_bone_project/notebook_flatfile_smoke_level7/tissue_sections

Config:
  subject_dir: /cis/home/dpadova/Documents/temporal_bone_project/data/FlatFiles
  output_dir: /cis/home/dpadova/Documents/temporal_bone_project/notebook_flatfile_smoke_level7/tissue_sections
  species_name: Macaque
  ext: .tif
  slice_down: 1
  res_down: 1
  max_slice: None
  dv: [35.0536384, 35.05385984, 16.0]
  space: right-inferior-posterior


## Step 2: Clean Up Previous JSON Sidecars (Optional)

`nuke_json_sidecars()` removes all `.json` files from the directory so we can
regenerate them cleanly.  Use `dry_run=True` to preview what would be deleted.

In [ ]:
deleted = nuke_json_sidecars(output_dir, dry_run=False)

## Step 3: Generate samples.tsv and JSON Sidecars

`set_up_hist_for_emlddmm()` runs three steps in sequence:

1. **Downsample** slices (skipped if `slice_down=1` and `res_down=1`)
2. **Generate `samples.tsv`** - lists all images, detects missing slices in the
   numbering sequence, and inserts placeholder rows marked `status=missing`
3. **Write JSON sidecars** - one `.json` per image with voxel size (`dv`) and
   anatomical coordinate space via `histsetup.generate_sidecars()`

In [ ]:
set_up_hist_for_emlddmm(histsetup_config)

## Step 4: Verify Outputs

In [ ]:
import json

# Show samples.tsv
tsv_path = output_dir / "samples.tsv"
if tsv_path.exists():
    lines = tsv_path.read_text().splitlines()
    print(f"samples.tsv ({len(lines) - 1} entries):")
    for line in lines[:11]:
        print(f"  {line}")
    if len(lines) > 11:
        print(f"  ... ({len(lines) - 11} more rows)")
else:
    print("samples.tsv not found.")

# Show JSON sidecars
jsons = sorted(output_dir.glob("*.json"))
print(f"\n{len(jsons)} JSON sidecar(s) generated")
for j in jsons[:5]:
    print(f"  {j.name}")
if len(jsons) > 5:
    print(f"  ... and {len(jsons) - 5} more")

# Pretty-print one sidecar
if jsons:
    print(f"\nContents of {jsons[0].name}:")
    print(json.dumps(json.loads(jsons[0].read_text()), indent=2))

## Step 5 (Optional): Generate samples.tsv Independently

If you only need the TSV (without JSON sidecars or downsampling), call
`make_samples_tsv()` directly.  This does **not** require the `emlddmm` package.

In [ ]:
tsv_path = make_samples_tsv(
    output_dir,
    species_name="Macaque",
    ext=".tif",
)
print(f"Written: {tsv_path}")

## Understanding the Output

### `samples.tsv` columns

| Column | Description |
|--------|-------------|
| `sample_id` | Image filename |
| `participant_id` | First 5 characters of the filename |
| `species` | Species label |
| `status` | `present` or `missing` |

Missing slices are inferred by comparing the observed slice numbers against the
expected arithmetic sequence.  For example, with `spacing=9` (every 10th slice),
expected indices are 1, 11, 21, 31, ...  Any gaps produce `missing` rows.

### JSON sidecar structure

Each `.json` file contains:
- `dv` - voxel dimensions `[dx, dy, dz]` in micrometers
- `space` - anatomical coordinate system (e.g. `"right-inferior-posterior"`)
- Additional metadata as needed by EM-LDDMM